---
short_title: JSON zu MARC-XML
numbering:
    heading_1: true
    heading_2: true
    title: true
---
# JSON-Daten in MARC-XML überführen

```{caution} Work In Progress
Diese Inhalte sind noch in Bearbeitung.
```

Nicht immer liegen bibliografische Daten bereits als MARC-XML vor. Häufig
stammen sie aus einem Web-Import, einer Excel-Tabelle oder einer eigenen
Erfassung im **{term}`JSON`**-Format. In diesem Notebook bilden wir solche
JSON-Quelldaten auf die MARC-Struktur ab und schreiben daraus eine valide
{term}`MARC-XML`-Datei – die typische **Import-Vorbereitung** für ein Bibliothekssystem.


## JSON-Quelldaten einlesen

Unsere Beispieldaten stammen aus dem Generator `generate_marc_data.py`. Wir
lesen sie mit dem Standardmodul `json` ein.


In [1]:
import json
from pathlib import Path


def finde_projektwurzel():
    """Sucht das Verzeichnis, das den Ordner assets/data enthält."""
    start = Path.cwd().resolve()
    for kandidat in [start, *start.parents]:
        if (kandidat / "assets" / "data").is_dir():
            return kandidat
    raise FileNotFoundError("Projektwurzel mit assets/data nicht gefunden.")


PROJEKTWURZEL = finde_projektwurzel()
DATEN = PROJEKTWURZEL / "assets" / "data"

with open(DATEN / "beispieldaten.json", encoding="utf-8") as datei:
    daten = json.load(datei)

print(f"{daten['anzahl_titel']} Titel geladen")
print("Stand:", daten["stand"])

18 Titel geladen
Stand: 2026-10-07


Ein einzelner Titel sieht so aus:

In [2]:
daten["titel"][0]

{'autor': 'Keller, Miriam',
 'titel': 'Datenkuration in wissenschaftlichen Bibliotheken',
 'untertitel': 'Praktiken, Werkzeuge und Standards',
 'ort': 'Berlin',
 'verlag': 'Wissenschaftsverlag Berlin',
 'jahr': 2021,
 'sprache': 'ger',
 'schlagwoerter': ['Bibliothekswissenschaft', 'Forschungsdaten', 'Metadaten'],
 'id': '000000001',
 'isbn': '9783110470017'}

## Das Mapping: JSON → MARC

Beim Mapping legen wir für jedes JSON-Feld fest, in welchem MARC-Feld und
Unterfeld es landet. Für unser Projekt sieht die Zuordnung so aus:

| JSON-Feld | MARC-Feld | Erläuterung |
| --- | --- | --- |
| `id` | `001` | Record-ID (Kontrollfeld) |
| `jahr`, `sprache`, `ort` | `008` | codierte Angaben (Fixfeld) |
| `isbn` | `020 $a` | ISBN |
| `sprache` | `041 $a` | Sprachcode |
| `autor` | `100 1# $a` | Hauptverantwortlichkeit |
| `titel`, `untertitel` | `245 10 $a $b` | Titel und Untertitel |
| `ort`, `verlag`, `jahr` | `264 #1 $a $b $c` | Erscheinungsangaben |
| `schlagwoerter` | `650 #7 $a` | ein Feld je Schlagwort |

Das Zeichen `#` steht hier für ein Leerzeichen als Indikator.


## Werkzeuge zum Aufbauen von XML

Bevor wir Records bauen, legen wir den MARC-Namensraum fest und definieren
Hilfsfunktionen. Wichtig ist `ET.register_namespace("", MARC_NS)`: Damit schreibt
`ElementTree` den Namensraum als Standard-Namensraum (`xmlns=...`) und nicht als
unschönes Präfix.


In [3]:
from xml.etree import ElementTree as ET

MARC_NS = "http://www.loc.gov/MARC21/slim"
ET.register_namespace("", MARC_NS)
NS_TAG = f"{{{MARC_NS}}}"

LEADER = "00000nam a2200000 a 4500"

LAENDERCODES = {
    "Berlin": "gw ",
    "München": "gw ",
    "Leipzig": "gw ",
    "Frankfurt": "gw ",
    "Göttingen": "gw ",
    "London": "enk",
    "New York": "nyu",
    "Paris": "fr ",
    "Amsterdam": "ne ",
    "Wien": "au ",
    "Zürich": "sz ",
    "Lisboa": "po ",
    "Roma": "it ",
    "Tokyo": "ja ",
    "Cape Town": "sa ",
    "Stockholm": "sw ",
    "Helsinki": "fi ",
}


def feld_008(jahr, sprache, ort):
    """Vereinfachtes, aber strukturell korrektes 008-Feld (40 Zeichen)."""
    land = LAENDERCODES.get(ort, "   ")
    return f"240101s{jahr:04d}    {land}{' ' * 17}{sprache} d"


def kontrollfeld(record, tag, wert):
    """Hängt ein Kontrollfeld an einen Record."""
    ET.SubElement(record, NS_TAG + "controlfield", {"tag": tag}).text = str(wert)


def datenfeld(record, tag, ind1, ind2, unterfelder):
    """Hängt ein Datenfeld mit Unterfeldern an einen Record."""
    feld = ET.SubElement(
        record, NS_TAG + "datafield", {"tag": tag, "ind1": ind1, "ind2": ind2}
    )
    for code, wert in unterfelder:
        ET.SubElement(feld, NS_TAG + "subfield", {"code": code}).text = str(wert)

## Einen Record aus JSON bauen

Nun übersetzen wir einen JSON-Eintrag in ein `<record>`-Element. Die Struktur
entspricht genau der Abbildungstabelle von oben.


In [4]:
def json_zu_record(eintrag):
    """Erzeugt aus einem JSON-Titel ein MARC-XML-Record-Element."""
    record = ET.Element(NS_TAG + "record")
    ET.SubElement(record, NS_TAG + "leader").text = LEADER

    kontrollfeld(record, "001", eintrag["id"])
    kontrollfeld(record, "003", "DE-101")
    kontrollfeld(record, "008", feld_008(eintrag["jahr"], eintrag["sprache"], eintrag["ort"]))

    datenfeld(record, "020", " ", " ", [("a", eintrag["isbn"])])
    datenfeld(record, "041", "0", " ", [("a", eintrag["sprache"])])
    datenfeld(record, "100", "1", " ", [("a", eintrag["autor"])])

    unterfelder_245 = [("a", eintrag["titel"])]
    if eintrag.get("untertitel"):
        unterfelder_245.append(("b", eintrag["untertitel"]))
    datenfeld(record, "245", "1", "0", unterfelder_245)

    datenfeld(
        record,
        "264",
        " ",
        "1",
        [
            ("a", eintrag["ort"]),
            ("b", eintrag["verlag"]),
            ("c", eintrag["jahr"]),
        ],
    )

    for schlagwort in eintrag["schlagwoerter"]:
        datenfeld(record, "650", " ", "7", [("a", schlagwort)])

    return record

## Die Collection schreiben

Alle Records kommen in eine `<collection>`. Mit `ET.indent()` wird die Ausgabe
lesbar eingerückt, `write()` speichert die Datei mit XML-Deklaration.


In [5]:
collection = ET.Element(NS_TAG + "collection")
for eintrag in daten["titel"]:
    collection.append(json_zu_record(eintrag))

baum = ET.ElementTree(collection)
ET.indent(baum, space="  ")

ziel = DATEN / "beispieldaten_import.marcxml"
baum.write(ziel, encoding="utf-8", xml_declaration=True)
print("geschrieben:", ziel.relative_to(PROJEKTWURZEL))

geschrieben: assets/data/beispieldaten_import.marcxml


## Kontrolle: Datei zurücklesen

Zum Abschluss lesen wir die erzeugte Datei wieder ein und prüfen, ob sie die
erwartete Anzahl Records enthält.


In [6]:
NAMENSRAUM = {"m": MARC_NS}
pruef_baum = ET.parse(ziel)
pruef_records = pruef_baum.getroot().findall("m:record", NAMENSRAUM)
print(f"Records in der Zieldatei: {len(pruef_records)}")

titel_feld = pruef_records[0].find("m:datafield[@tag='245']", NAMENSRAUM)
print("Titel des ersten Records:",
      titel_feld.find("m:subfield[@code='a']", NAMENSRAUM).text)

Records in der Zieldatei: 18
Titel des ersten Records: Datenkuration in wissenschaftlichen Bibliotheken


## Zusammenfassung

Sie haben JSON-Daten über ein festes Mapping in die MARC-Struktur überführt und
als valides MARC-XML gespeichert. Genau dieser Schritt entspricht der
Import-Vorbereitung in der Praxis. Im nächsten Notebook exportieren wir
MARC-Daten nach **BibTeX**.

```{seealso}
Weiter geht es mit dem [Export nach BibTeX](./040-Export_BibTeX.ipynb).
```
